# Inference & Demo — SeribuCerita Emotion Classifier

**Capstone Project CC26-PSU212 — AI Path**

Notebook ini mendemonstrasikan inference model klasifikasi emosi, menguji robustness pada edge case, dan menyediakan chatbot interaktif.

### Prasyarat
- Model `.keras` dan tokenizer dari notebook training
- GPU direkomendasikan (opsional)

### Struktur
1. Setup & load model
2. Fungsi inference
3. Contoh prediksi
4. Robustness testing (negasi, sarkasme, narasi)
5. Interactive chatbot

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==4.44.0', 'tf-keras'])
print('Dependencies installed. Restart kernel if needed, then run from next cell.')

In [ ]:
import os
import numpy as np
import tensorflow as tf
os.environ['TF_USE_LEGACY_KERAS'] = '1'
from transformers import BertTokenizer, TFBertModel

print(f'TensorFlow {tf.__version__} — GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# Path ke model artifacts directory yang berisi saved_model/, tokenizer/, dan model_final.keras
MODEL_DIR = './output'  # Path to model artifacts (adjust to your environment)

KERAS_MODEL_PATH = f'{MODEL_DIR}/model_final.keras'
SAVED_MODEL_PATH = f'{MODEL_DIR}/saved_model'
TOKENIZER_DIR    = f'{MODEL_DIR}/tokenizer'
MAX_LENGTH = 128

ID2LABEL = {0: 'anger', 1: 'fear', 2: 'sad', 3: 'neutral', 4: 'happy'}
LABEL_EMOJI = {'anger': '😠', 'fear': '😨', 'sad': '😢', 'neutral': '😐', 'happy': '😄'}

for p in [KERAS_MODEL_PATH, TOKENIZER_DIR, SAVED_MODEL_PATH]:
    print(f'  {"✓" if os.path.exists(p) else "✗ MISSING"} {p}')

## 2. Custom Components

Custom layers dan loss function harus didefinisikan sebelum load model `.keras`.

In [ ]:
class AttentionPoolingLayer(tf.keras.layers.Layer):
    """Attention-weighted pooling over BERT hidden states."""
    def __init__(self, hidden_size=768, **kwargs):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.attention_dense = tf.keras.layers.Dense(1)

    def call(self, hidden_states, attention_mask):
        scores = tf.squeeze(self.attention_dense(hidden_states), axis=-1)
        scores = scores + (1.0 - tf.cast(attention_mask, tf.float32)) * -1e9
        weights = tf.expand_dims(tf.nn.softmax(scores, axis=-1), axis=-1)
        return tf.reduce_sum(hidden_states * weights, axis=1)

    def get_config(self):
        return {**super().get_config(), 'hidden_size': self.hidden_size}


class FocalLossWithSmoothing(tf.keras.losses.Loss):
    """Focal loss with label smoothing and optional class weights."""
    def __init__(self, num_classes=5, gamma=2.0, label_smoothing=0.05,
                 class_weights=None, name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.class_weights = class_weights

    def call(self, y_true, y_pred):
        y_true_int = tf.cast(y_true, tf.int32)
        y_smooth = tf.one_hot(y_true_int, self.num_classes) * (1.0 - self.label_smoothing) + self.label_smoothing / self.num_classes
        probs = tf.clip_by_value(tf.nn.softmax(y_pred, axis=-1), 1e-7, 1.0 - 1e-7)
        focal = tf.pow(1.0 - probs, self.gamma) * (-y_smooth * tf.math.log(probs))
        per_sample = tf.reduce_sum(focal, axis=-1)
        if self.class_weights is not None:
            per_sample *= tf.gather(tf.constant(self.class_weights, dtype=tf.float32), y_true_int)
        return tf.reduce_mean(per_sample)

    def get_config(self):
        return {**super().get_config(), 'num_classes': self.num_classes,
                'gamma': self.gamma, 'label_smoothing': self.label_smoothing,
                'class_weights': self.class_weights}


class EmotionClassifier(tf.keras.Model):
    """IndoBERT + Attention Pooling + Dense classifier."""
    def __init__(self, model_name, num_classes, dropout=0.2, **kwargs):
        super().__init__(**kwargs)
        self.model_name = model_name
        self.num_classes = num_classes
        self.dropout_rate = dropout
        self.bert = TFBertModel.from_pretrained(model_name, name='bert')
        self.pooler = AttentionPoolingLayer(hidden_size=768, name='attention_pooler')
        self.dropout = tf.keras.layers.Dropout(dropout, name='dropout')
        self.classifier = tf.keras.layers.Dense(num_classes, name='classifier')

    def call(self, inputs, training=False):
        bert_out = self.bert(input_ids=inputs['input_ids'],
                            attention_mask=inputs['attention_mask'], training=training)
        pooled = self.pooler(bert_out.last_hidden_state, inputs['attention_mask'])
        return self.classifier(self.dropout(pooled, training=training))

    def get_config(self):
        return {**super().get_config(), 'model_name': self.model_name,
                'num_classes': self.num_classes, 'dropout': self.dropout_rate}

print('Custom components defined.')

## 3. Load Model

In [ ]:
tokenizer = BertTokenizer.from_pretrained(TOKENIZER_DIR)

model = tf.keras.models.load_model(KERAS_MODEL_PATH, custom_objects={
    'AttentionPoolingLayer': AttentionPoolingLayer,
    'FocalLossWithSmoothing': FocalLossWithSmoothing,
    'EmotionClassifier': EmotionClassifier,
})

total_params = sum(np.prod(v.shape) for v in model.variables)
print(f'Model loaded — {total_params:,} parameters')

## 4. Inference

In [ ]:
def predict_emotion(text):
    """Predict emotion label and confidence scores for Indonesian text."""
    enc = tokenizer(text, max_length=MAX_LENGTH, truncation=True,
                    padding='max_length', return_tensors='np')
    logits = model({'input_ids': tf.constant(enc['input_ids']),
                    'attention_mask': tf.constant(enc['attention_mask'])}, training=False)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[0]
    pred_id = int(np.argmax(probs))
    scores = sorted([(ID2LABEL[i], float(probs[i])) for i in range(len(probs))],
                    key=lambda x: x[1], reverse=True)
    return {'label': ID2LABEL[pred_id], 'confidence': float(probs[pred_id]), 'scores': scores}


def display_result(text, result):
    """Pretty-print prediction with confidence bars."""
    label, conf = result['label'], result['confidence']
    bar = '█' * int(conf * 30) + '░' * (30 - int(conf * 30))
    print(f'\n  Text   : "{text}"')
    print(f'  Result : {LABEL_EMOJI[label]} {label.upper()} ({conf*100:.1f}%)')
    print(f'  [{bar}]')
    for lbl, score in result['scores']:
        mini = '█' * int(score * 20)
        mark = ' ◄' if lbl == label else ''
        print(f'    {LABEL_EMOJI[lbl]} {lbl:8s} {score*100:5.1f}% {mini}{mark}')
    print()

## 5. Example Predictions

In [ ]:
examples = [
    'Aku sedih banget hari ini, semua kerjaan numpuk',
    'Bahagia banget, akhirnya lulus skripsi!',
    'Marah-marah terus, kerjaan ga selesai-selesai',
    'Takut banget besok presentasi di depan banyak orang',
    'Hari ini cuaca cerah, biasa aja',
    'Sumpah kesel banget sama dia, ga tau malu',
    'Kangen banget sama keluarga di rumah',
]

for text in examples:
    display_result(text, predict_emotion(text))

## 6. Robustness Testing

Menguji model pada edge case di luar distribusi training.

In [ ]:
def run_test(name, texts, expected):
    """Run a test group and print results."""
    print(f'\n{"="*55}\n  {name}\n{"="*55}')
    correct = 0
    for text, exp in zip(texts, expected):
        r = predict_emotion(text)
        ok = r['label'] == exp
        correct += ok
        mark = '✓' if ok else f'✗ (expected {exp})'
        print(f"  [{LABEL_EMOJI[r['label']]} {r['label']:8s} {r['confidence']*100:5.1f}%] {mark}")
        print(f'    "{text}"')
    print(f'\n  Accuracy: {correct}/{len(expected)} ({correct/len(expected)*100:.0f}%)')

In [ ]:
# Negasi — apakah model mengenali "tidak marah" ≠ "marah"?
pairs = [
    ('aku sedih',   'aku tidak sedih'),
    ('aku marah',   'aku ga marah kok'),
    ('aku takut',   'aku bukan takut'),
    ('aku bahagia', 'aku ga bahagia sama sekali'),
]
print(f'{"="*55}\n  Negasi\n{"="*55}')
for pos, neg in pairs:
    r1, r2 = predict_emotion(pos), predict_emotion(neg)
    flag = '⚠️ same' if r1['label'] == r2['label'] else '✓ changed'
    print(f'\n  + "{pos}" → {LABEL_EMOJI[r1["label"]]} {r1["label"]} ({r1["confidence"]*100:.0f}%)')
    print(f'  - "{neg}" → {LABEL_EMOJI[r2["label"]]} {r2["label"]} ({r2["confidence"]*100:.0f}%) {flag}')

In [ ]:
run_test('Sarkasme', [
    'seneng banget dong di PHP-in lagi, makasih ya',
    'mantap banget hidup gue, gagal mulu',
    'wah bagus banget nilai segitu, emang pinter',
    'asik banget ditinggal pas lagi butuh',
], expected=['sad', 'sad', 'anger', 'sad'])

In [ ]:
run_test('Narasi Cerita', [
    'Si Budi menangis di sudut kamar setelah mendengar kabar itu',
    'Matanya berbinar melihat hadiah kejutan dari sahabatnya',
    'Tangannya gemetar, jantungnya berdegup menunggu pengumuman',
    'Dia membanting pintu sekeras mungkin, amarah memuncak',
    'Mereka hanya duduk diam, menikmati sore yang tenang',
], expected=['sad', 'happy', 'fear', 'anger', 'neutral'])

## 7. Interactive Chatbot

Ketik teks Bahasa Indonesia → lihat prediksi emosi real-time. Ketik `quit` untuk keluar.

In [ ]:
print('=' * 55)
print('🎯 SeribuCerita Emotion Classifier')
print('   😠 anger | 😨 fear | 😢 sad | 😐 neutral | 😄 happy')
print('   Ketik "quit" untuk keluar')
print('=' * 55)

while True:
    try:
        text = input('\nKamu: ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not text:
        continue
    if text.lower() in ('quit', 'exit', 'q', 'keluar'):
        print('Sampai jumpa! 👋')
        break
    result = predict_emotion(text)
    display_result(text, result)